In [ ]:
#@title ① config — EDIT THIS CELL

# --- Gemini API key
# Option A: set GEMINI_API_KEY as an environment variable before launching Jupyter
# Option B: paste it directly here (do NOT commit this file if you do this)
GEMINI_API_KEY = ""   # leave empty to read from env var

# --- Episode to explore ---
EPISODE_IDX = 0

# --- Rendered image size ---
RENDER_WIDTH  = 224
RENDER_HEIGHT = 224

# --- FPS of the dataset (D4RL kitchen ~10 Hz) ---
DATA_FPS = 10

# --- Frame sampling rate sent to Gemini ---
# Higher = more frames = more detail for VLM, costs more tokens.
GEMINI_VIDEO_FPS = 5

# --- Gemini model ---
GEMINI_MODEL = "gemini-2.5-pro-preview-05-06"

# --- Prompt (edit freely — this is what you iterate on) ---
KITCHEN_PROMPT = """
Role: Robotics Data Specialist
Task: Segment a robot kitchen manipulation demonstration into a sequence of discrete options.

## Video Context ##
A Franka robot arm (9 DoF) is performing kitchen manipulation tasks in a MuJoCo simulation.
Each demonstration completes exactly 4 subtasks in some order.

**The 4 tasks and what they look like:**
- **Microwave**: The robot pushes open the microwave door (it swings open on a hinge).
- **Kettle**: The robot grasps and physically relocates a cylindrical kettle on the stovetop.
  The kettle visibly moves to a new position.
- **Burner**: The robot reaches the stove panel and turns an oven knob. When the knob is
  turned successfully, a white flame/heat visual appears under the corresponding burner.
- **Cabinet**: The robot slides a cabinet door horizontally open.

**Option Definitions:**
1. **TRANSIT**: Robot arm moving through free space. No object is being contacted or manipulated.
2. **MICROWAVE**: Robot reaching toward or actively pushing the microwave door open.
3. **KETTLE**: Robot reaching toward, grasping, or relocating the kettle.
4. **BURNER**: Robot reaching toward or turning the oven knob (white flame appears under burner).
5. **CABINET**: Robot reaching toward or sliding the cabinet door open.

**Frame Number Tracking:**
Each frame has a red number burned into the top-left corner. Read these directly.

**Processing Workflow:**
1. Watch the full video and identify the order the 4 tasks are performed.
2. Write chain-of-thought with exact frame ranges for each option.
3. Self-check: every frame must belong to exactly one option. No gaps, no overlaps.
4. Output the JSON array.

**Output Format:**
Two sections separated by `---`. First: chain of thought + frame ranges. Second: JSON only.

### Example Output:

Chain of thought:
Robot starts in free space (TRANSIT), then opens the microwave (MICROWAVE).
Pulls back (TRANSIT), moves the kettle (KETTLE).
Returns to free space (TRANSIT), turns the burner knob — white flame appears (BURNER).
Finally slides the cabinet open (CABINET).

TRANSIT: 0-18
MICROWAVE: 18-55
TRANSIT: 55-74
KETTLE: 74-121
TRANSIT: 121-138
BURNER: 138-167
TRANSIT: 167-180
CABINET: 180-212

---
[
  {"option": "TRANSIT",   "start": 0,   "end": 18},
  {"option": "MICROWAVE", "start": 18,  "end": 55},
  {"option": "TRANSIT",   "start": 55,  "end": 74},
  {"option": "KETTLE",    "start": 74,  "end": 121},
  {"option": "TRANSIT",   "start": 121, "end": 138},
  {"option": "BURNER",    "start": 138, "end": 167},
  {"option": "TRANSIT",   "start": 167, "end": 180},
  {"option": "CABINET",   "start": 180, "end": 212}
]
"""

In [4]:
#@title ② imports + Gemini setup
import os, json, time
import numpy as np
import cv2
import matplotlib.pyplot as plt
import mediapy as media
from IPython.display import display, Markdown
from google import genai
from google.genai import types
import gymnasium as gym
import gymnasium_robotics
import minari

gym.register_envs(gymnasium_robotics)

# API key: use value from config cell if set, otherwise read from environment
api_key = GEMINI_API_KEY or os.environ.get('GEMINI_API_KEY', '')
assert api_key, "Set GEMINI_API_KEY in cell ① or as an environment variable"
os.environ['GEMINI_API_KEY'] = api_key
client = genai.Client()
print('Gemini client ready.')

Gemini client ready.


AdroitHandRelocateDense-v1, AdroitHandHammerDense-v1, AdroitHandDoorDense-v1 environment's reward functions were updated in v1.2.1 without an environment version update. Therefore, use gymnasium-robotics==1.2.0 for v1 reproducibility or use v2 in gymnasium-robotics>=1.4.3. See https://github.com/Farama-Foundation/Gymnasium-Robotics/pull/220 for more details


In [5]:
#@title ③ load dataset + renderer
# Downloads ~50MB from HuggingFace on first run, cached locally after that.

print('Loading D4RL kitchen dataset...')
minari_dataset = minari.load_dataset('D4RL/kitchen/complete-v2', download=True)
episodes = list(minari_dataset.iterate_episodes())
print(f'Loaded {len(episodes)} episodes  |  '
      f'obs_dim={episodes[0].observations["observation"].shape[1]}  |  '
      f'action_dim={episodes[0].actions.shape[1]}  |  '
      f'avg_len={int(np.mean([len(e.actions) for e in episodes]))} steps')

print('\nCreating FrankaKitchen-v1 render environment...')
render_env = gym.make('FrankaKitchen-v1', render_mode='rgb_array',
                      width=RENDER_WIDTH, height=RENDER_HEIGHT)
print('Renderer ready.')

Loading D4RL kitchen dataset...
Loaded 19 episodes  |  obs_dim=59  |  action_dim=9  |  avg_len=221 steps

Creating FrankaKitchen-v1 render environment...
Renderer ready.


In [ ]:
#@title ④ helper functions

OPTION_NAMES  = ['TRANSIT', 'MICROWAVE', 'KETTLE', 'BURNER', 'CABINET']
OPTION_COLORS = {
    'TRANSIT':  (128, 128, 128),
    'MICROWAVE':(255,  60,  60),
    'KETTLE':   ( 60, 200,  60),
    'BURNER':   (255, 165,   0),
    'CABINET':  (120,  80, 200),
}

def load_episode(idx):
    """Render episode by replaying recorded actions. Returns (images, states, actions)."""
    ep = episodes[idx]
    render_env.reset()
    frames = []
    for action in ep.actions:
        render_env.step(action)
        frames.append(render_env.render().astype(np.uint8))
    return np.array(frames), ep.observations['observation'], ep.actions

def burn_frame_number(frame, idx):
    out = frame.copy()
    h, w = out.shape[:2]
    cv2.putText(out, str(idx), (4, int(14 * max(0.4, w/200))),
                cv2.FONT_HERSHEY_SIMPLEX, max(0.4, w/200), (255, 0, 0), 1, cv2.LINE_AA)
    return out

def burn_option_overlay(frame, option_name):
    out = frame.copy()
    color = OPTION_COLORS.get(option_name, (255, 255, 255))
    h, w = out.shape[:2]
    cv2.rectangle(out, (0, 0), (w-1, h-1), color, 3)
    cv2.putText(out, option_name, (4, h-6),
                cv2.FONT_HERSHEY_SIMPLEX, max(0.35, w/250), color, 1, cv2.LINE_AA)
    return out

def frames_to_mp4(frames, fps, path, burn_numbers=True, option_labels=None):
    h, w = frames[0].shape[:2]
    writer = cv2.VideoWriter(path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
    for i, f in enumerate(frames):
        out = burn_frame_number(f, i) if burn_numbers else f.copy()
        if option_labels is not None and i < len(option_labels):
            out = burn_option_overlay(out, option_labels[i])
        writer.write(cv2.cvtColor(out, cv2.COLOR_RGB2BGR))
    writer.release()

def play_episode(idx, height=400):
    """Render and display episode inline."""
    print(f'Rendering episode {idx} ({len(episodes[idx].actions)} steps)...')
    images, states, actions = load_episode(idx)
    print(f'Done. {len(images)} frames rendered.')
    tmp = f'_ep{idx}_raw.mp4'
    frames_to_mp4(list(images), fps=DATA_FPS, path=tmp)
    media.show_video(media.read_video(tmp), height=height)
    os.remove(tmp)
    return images, states, actions

def segments_to_frame_labels(segments, n_frames):
    labels = ['TRANSIT'] * n_frames
    for seg in segments:
        for i in range(seg['start'], min(seg['end'], n_frames)):
            labels[i] = seg['option']
    return labels

def parse_json_from_response(text):
    part = text.split('---')[-1].strip().replace('```json','').replace('```','').strip()
    return json.loads(part)

print('Helper functions loaded.')

In [7]:
#@title ⑤ play raw episode
# Change EPISODE_IDX in cell ① then re-run this cell.
cached_images, cached_states, cached_actions = play_episode(EPISODE_IDX)

Rendering episode 0 (212 steps)...
Done. 212 frames rendered.


This browser does not support the video tag.


In [ ]:
#@title ⑥ label episode with Gemini
# Edit KITCHEN_PROMPT in cell ①, then re-run this cell only — no re-rendering needed.

def label_episode_gemini(images, idx, prompt, model, video_fps):
    tmp = f'_label_ep{idx}.mp4'
    print(f'Compiling {len(images)}-frame video...')
    frames_to_mp4(list(images), fps=DATA_FPS, path=tmp)
    try:
        print('Uploading to Gemini...')
        cf = client.files.upload(file=tmp)
        while cf.state.name == 'PROCESSING':
            print('.', end='', flush=True)
            time.sleep(4)
            cf = client.files.get(name=cf.name)
        if cf.state.name == 'FAILED':
            raise RuntimeError('Gemini file processing failed.')
        print(f'\nCalling {model}...')
        response = client.models.generate_content(
            model=model,
            contents=types.Content(parts=[
                types.Part(
                    file_data=types.FileData(file_uri=cf.uri, mime_type='video/mp4'),
                    video_metadata=types.VideoMetadata(fps=video_fps)
                ),
                types.Part(text=prompt)
            ]),
            config=types.GenerateContentConfig(
                temperature=0.0,
                thinking_config=types.ThinkingConfig(include_thoughts=True, thinking_budget=4000),
            )
        )
        print('\n--- Gemini Response ---')
        display(Markdown(response.text))
        u = response.usage_metadata
        print(f'Tokens — prompt:{u.prompt_token_count} thoughts:{u.thoughts_token_count} output:{u.candidates_token_count}')
        return response.text, len(images)
    finally:
        if 'cf' in locals(): client.files.delete(name=cf.name)
        if os.path.exists(tmp): os.remove(tmp)

response_text, n_frames = label_episode_gemini(
    cached_images, EPISODE_IDX, KITCHEN_PROMPT, GEMINI_MODEL, GEMINI_VIDEO_FPS
)

In [ ]:
#@title ⑦ visualize labeled episode

try:
    segments = parse_json_from_response(response_text)
    print('Parsed segments:')
    for s in segments:
        print(f"  {s['option']:15s}  frames {s['start']:4d} – {s['end']:4d}  ({s['end']-s['start']} frames)")
except Exception as e:
    print(f'Could not parse JSON: {e}')
    print('Paste the JSON array into segments manually and re-run.')
    segments = []  # <- paste manually if needed

if segments:
    labels = segments_to_frame_labels(segments, len(cached_images))
    tmp = f'_labeled_ep{EPISODE_IDX}.mp4'
    frames_to_mp4(list(cached_images), fps=DATA_FPS, path=tmp, option_labels=labels)
    media.show_video(media.read_video(tmp), height=400)
    os.remove(tmp)

In [ ]:
#@title ⑧ option distribution

if segments:
    from collections import Counter
    counts = Counter(labels)
    total  = sum(counts.values())
    print(f"{'Option':<15} {'Frames':>8} {'%':>8}")
    print('-' * 35)
    for name in OPTION_NAMES:
        n = counts.get(name, 0)
        print(f"{name:<15} {n:>8}  {100*n/total:>6.1f}%")
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.bar(OPTION_NAMES,
           [counts.get(n, 0) for n in OPTION_NAMES],
           color=[tuple(c/255 for c in OPTION_COLORS[n]) for n in OPTION_NAMES],
           edgecolor='black')
    ax.set_ylabel('Frames')
    ax.set_title(f'Option distribution — Episode {EPISODE_IDX}')
    plt.tight_layout()
    plt.show()